# Text Preprocessing Pipeline

This notebook builds the preprocessing pipeline for sentiment analysis.

Goals:

- Clean raw text
- Build a custom tokenizer
- Remove stopwords
- Apply stemming
- Analyze vocabulary reduction

The output of this notebook will be cleaned text that can later be converted into TF-IDF vectors and used for sentiment classification.

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PROJECT_ROOT = "/content/drive/MyDrive/emotion_sentiment_fusion_detector"

df = pd.read_csv(
    f"{PROJECT_ROOT}/data/imdb/imdb_processed.csv"
)

df.head()

,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


## Inspect Raw Review

Before preprocessing, inspect what raw text looks like.

In [6]:
print(df.iloc[0]["review"])

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac

## Step 1: Text Cleaning

Remove:

- HTML tags
- punctuation
- numbers
- extra spaces

Convert text to lowercase.

In [7]:
def clean_text(text):

    text = str(text)

    text = text.lower()

    text = re.sub(r"<.*?>", " ", text)

    text = re.sub(r"[^a-z\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [8]:
sample = df.iloc[0]["review"]

print("ORIGINAL:\n")
print(sample[:500])

print("\n" + "="*50 + "\n")

print("CLEANED:\n")
print(clean_text(sample)[:500])

ORIGINAL:

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ


CLEANED:

one of the other reviewers has mentioned that after watching just oz episode you ll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the word it is called oz as that is

## Step 2: Custom Tokenizer

A tokenizer splits text into individual words.

We will implement a simple tokenizer ourselves.

In [9]:
def tokenize(text):

    return text.split()

In [10]:
tokens = tokenize(
    clean_text(df.iloc[0]["review"])
)

tokens[:50]

['one',
 'of',
 'the',
 'other',
 'reviewers',
 'has',
 'mentioned',
 'that',
 'after',
 'watching',
 'just',
 'oz',
 'episode',
 'you',
 'll',
 'be',
 'hooked',
 'they',
 'are',
 'right',
 'as',
 'this',
 'is',
 'exactly',
 'what',
 'happened',
 'with',
 'me',
 'the',
 'first',
 'thing',
 'that',
 'struck',
 'me',
 'about',
 'oz',
 'was',
 'its',
 'brutality',
 'and',
 'unflinching',
 'scenes',
 'of',
 'violence',
 'which',
 'set',
 'in',
 'right',
 'from',
 'the']

## Step 3: Stopword Removal

Stopwords are very common words that often carry little sentiment information.

Examples:

- the
- is
- and
- of
- to

In [11]:
stopwords = {
    "the",
    "a",
    "an",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "of",
    "to",
    "in",
    "on",
    "for",
    "with",
    "that",
    "this",
    "it",
    "as",
    "at",
    "by",
    "from",
    "or",
    "and",
    "but"
}

In [12]:
def remove_stopwords(tokens):

    return [
        word
        for word in tokens
        if word not in stopwords
    ]

In [13]:
filtered_tokens = remove_stopwords(
    tokens
)

filtered_tokens[:50]

['one',
 'other',
 'reviewers',
 'has',
 'mentioned',
 'after',
 'watching',
 'just',
 'oz',
 'episode',
 'you',
 'll',
 'hooked',
 'they',
 'right',
 'exactly',
 'what',
 'happened',
 'me',
 'first',
 'thing',
 'struck',
 'me',
 'about',
 'oz',
 'its',
 'brutality',
 'unflinching',
 'scenes',
 'violence',
 'which',
 'set',
 'right',
 'word',
 'go',
 'trust',
 'me',
 'not',
 'show',
 'faint',
 'hearted',
 'timid',
 'show',
 'pulls',
 'no',
 'punches',
 'regards',
 'drugs',
 'sex',
 'violence']

## Step 4: Stemming

Stemming reduces words to their root forms.

Examples:

playing → play

played → play

plays → play

This reduces vocabulary size.

In [14]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

In [15]:
def stem_tokens(tokens):

    return [
        stemmer.stem(word)
        for word in tokens
    ]

In [16]:
stemmed_tokens = stem_tokens(
    filtered_tokens
)

stemmed_tokens[:50]

['one',
 'other',
 'review',
 'ha',
 'mention',
 'after',
 'watch',
 'just',
 'oz',
 'episod',
 'you',
 'll',
 'hook',
 'they',
 'right',
 'exactli',
 'what',
 'happen',
 'me',
 'first',
 'thing',
 'struck',
 'me',
 'about',
 'oz',
 'it',
 'brutal',
 'unflinch',
 'scene',
 'violenc',
 'which',
 'set',
 'right',
 'word',
 'go',
 'trust',
 'me',
 'not',
 'show',
 'faint',
 'heart',
 'timid',
 'show',
 'pull',
 'no',
 'punch',
 'regard',
 'drug',
 'sex',
 'violenc']

## Full Preprocessing Pipeline

In [17]:
def preprocess_text(text):

    cleaned = clean_text(text)

    tokens = tokenize(cleaned)

    tokens = remove_stopwords(tokens)

    tokens = stem_tokens(tokens)

    return tokens

In [18]:
processed_example = preprocess_text(
    df.iloc[0]["review"]
)

processed_example[:100]

['one',
 'other',
 'review',
 'ha',
 'mention',
 'after',
 'watch',
 'just',
 'oz',
 'episod',
 'you',
 'll',
 'hook',
 'they',
 'right',
 'exactli',
 'what',
 'happen',
 'me',
 'first',
 'thing',
 'struck',
 'me',
 'about',
 'oz',
 'it',
 'brutal',
 'unflinch',
 'scene',
 'violenc',
 'which',
 'set',
 'right',
 'word',
 'go',
 'trust',
 'me',
 'not',
 'show',
 'faint',
 'heart',
 'timid',
 'show',
 'pull',
 'no',
 'punch',
 'regard',
 'drug',
 'sex',
 'violenc',
 'it',
 'hardcor',
 'classic',
 'use',
 'word',
 'call',
 'oz',
 'nicknam',
 'given',
 'oswald',
 'maximum',
 'secur',
 'state',
 'penitentari',
 'focus',
 'mainli',
 'emerald',
 'citi',
 'experiment',
 'section',
 'prison',
 'where',
 'all',
 'cell',
 'have',
 'glass',
 'front',
 'face',
 'inward',
 'so',
 'privaci',
 'not',
 'high',
 'agenda',
 'em',
 'citi',
 'home',
 'mani',
 'aryan',
 'muslim',
 'gangsta',
 'latino',
 'christian',
 'italian',
 'irish',
 'more',
 'so',
 'scuffl',
 'death',
 'stare']

## Apply Pipeline to Dataset

In [19]:
sample_df = df.head(5000).copy()

In [20]:
sample_df["tokens"] = sample_df["review"].apply(
    preprocess_text
)

In [22]:
sample_df[
    ["review", "tokens"]
].head()

,review,tokens
0,One of the other reviewers has mentioned that ...,"[one, other, review, ha, mention, after, watch..."
1,A wonderful little production. <br /><br />The...,"[wonder, littl, product, film, techniqu, veri,..."
2,I thought this was a wonderful way to spend ti...,"[i, thought, wonder, way, spend, time, too, ho..."
3,Basically there's a family where a little boy ...,"[basic, there, s, famili, where, littl, boy, j..."
4,"Petter Mattei's ""Love in the Time of Money"" is...","[petter, mattei, s, love, time, money, visual,..."


## Vocabulary Analysis

Measure how preprocessing affects vocabulary size.

In [23]:
raw_words = []

for review in sample_df["review"]:

    raw_words.extend(
        str(review).split()
    )

raw_vocab = len(set(raw_words))

print(raw_vocab)

101552


In [24]:
processed_words = []

for tokens in sample_df["tokens"]:

    processed_words.extend(tokens)

processed_vocab = len(
    set(processed_words)
)

print(processed_vocab)

26181


In [25]:
print("Raw Vocabulary:", raw_vocab)
print("Processed Vocabulary:", processed_vocab)

Raw Vocabulary: 101552
Processed Vocabulary: 26181


In [26]:
word_freq = Counter(
    processed_words
)

word_freq.most_common(20)

[('i', 17359),
 ('s', 12874),
 ('movi', 10512),
 ('film', 9527),
 ('you', 7192),
 ('t', 6734),
 ('have', 6166),
 ('not', 6011),
 ('hi', 5720),
 ('he', 5658),
 ('one', 5503),
 ('all', 4650),
 ('they', 4522),
 ('like', 4491),
 ('who', 4227),
 ('so', 4124),
 ('there', 3709),
 ('just', 3570),
 ('about', 3456),
 ('her', 3447)]

# Text Preprocessing Summary

Steps implemented:

- Lowercasing
- HTML removal
- Punctuation removal
- Tokenization
- Stopword removal
- Stemming

Benefits:

- Reduced vocabulary size
- Reduced noise
- Standardized word forms
- Improved feature quality

The processed text is now ready for numerical representation using TF-IDF.